In [ ]:
import librosa
import numpy as np
import os

In [ ]:
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=22050, mono=True)
        
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs.T, axis=0) 
        
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        cent_mean = np.mean(cent)
        
        return np.hstack((mfccs_mean, zcr_mean, cent_mean))
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [ ]:
audio_data_paths = {
    'kick': './drum_set/kick/',
    'snare': './drum_set/snare/'
}

X = [] 
y = []

for label, folder_path in audio_data_paths.items():
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.wav'):
            file_path = os.path.join(folder_path, file_name)
            
            features = extract_features(file_path)
            
            if features is not None:
                X.append(features)
                y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Forma della matrice X: {X.shape}") # Sarà (Numero_di_file, 15)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# 1. Divisione in Train e Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. SCALING (Fondamentale!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Addestramento SVM
svm_model = SVC(kernel='rbf') # RBF di solito funziona benissimo con l'audio
svm_model.fit(X_train_scaled, y_train)

# Valutazione
accuracy = svm_model.score(X_test_scaled, y_test)
print(f"Accuratezza del modello: {accuracy * 100:.2f}%")